# Figure 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from plot_utils import check_data, load_integrated_table, table
check_data()


## Panel A

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D

CSV = Path("figure4_routing_backbone_ONE_CSV.csv")
df = load_integrated_table("figure4_routing_backbone_ONE_CSV.csv", low_memory=False)

PALETTE = {
    "flow":  "#2B4C7E",
    "outer": "#467CB9",
    "inner": "#8FB9B2",
    "grid":  "#D8DEE9",
}

boundary = df[df["table"].eq("boundary")].copy()
bounds = df[df["table"].eq("bounds")].iloc[0]
coords = df[df["table"].eq("coords")].copy()
corridors = df[
    (df["table"].eq("gross_corridors")) &
    (pd.to_numeric(df["flow_twh_it"], errors="coerce") > 0)
].copy().sort_values("flow_twh_it")
service = df[df["table"].eq("province_balance")].copy()

known = set(coords["province"].dropna())
corridors = corridors[
    corridors["source"].isin(known) &
    corridors["target"].isin(known)
].copy()

fig, ax = plt.subplots(figsize=(7.2, 8.7))

fig.patch.set_alpha(0)
ax.set_facecolor("none")

ax.set_xlim(float(bounds["xmin"]), float(bounds["xmax"]))
ax.set_ylim(float(bounds["ymin"]), float(bounds["ymax"]))

ax.set_aspect("equal", adjustable="box", anchor="NW")
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)
    spine.set_edgecolor("#00000000")

segments = []
for _, seg_df in boundary.groupby("segment_id"):
    seg_df = seg_df.sort_values("point_id")
    segments.append(seg_df[["map_x", "map_y"]].to_numpy())

ax.add_collection(LineCollection(
    segments,
    colors="#00000000",
    linewidths=0.72,
    alpha=0,
    zorder=0
))

ax.set_title(
    'a',
    loc="left",
    fontweight="bold",
    fontsize=8.5,
    color="#1F2D3A"
)

ax.scatter(
    coords["map_x"],
    coords["map_y"],
    s=12,
    c="#00000000",
    edgecolor="white",
    linewidth=0.3,
    zorder=1,
    alpha=0.0
)

coord_lookup = coords.set_index("province")

max_flow = max(float(corridors["flow_twh_it"].max()), 1e-9)

for rank, (_, row) in enumerate(corridors.iterrows()):
    src = coord_lookup.loc[row["source"]]
    dst = coord_lookup.loc[row["target"]]

    strength = float(row["flow_twh_it"]) / max_flow
    rad = 0.08 if rank % 2 == 0 else -0.08

    ax.add_patch(FancyArrowPatch(
        (src["map_x"], src["map_y"]),
        (dst["map_x"], dst["map_y"]),
        arrowstyle="-|>",
        connectionstyle=f"arc3,rad={rad}",
        mutation_scale=7.4 + 7.2 * strength,
        lw=1.25 + 4.75 * strength,
        color=PALETTE["flow"],
        alpha=0.36 + 0.38 * strength,
        zorder=2
    ))

service = service.drop(columns=["map_x", "map_y", "lon", "lat"], errors="ignore")
service = service.merge(
    coords[["province", "province_en", "map_x", "map_y"]],
    on=["province", "province_en"],
    how="left"
).dropna(subset=["map_x", "map_y"])

total = np.maximum(service["executed_supply_twh_it"].astype(float), 0.0)
local = np.maximum(service["local_processed_twh_it"].astype(float), 0.0)

max_total = max(float(total.max()), 1e-9)

outer_size = 70.0 + 820.0 * np.sqrt(total / max_total)
inner_size = outer_size * np.where(total > 0, local / total, 0.0)

ax.scatter(
    service["map_x"],
    service["map_y"],
    s=outer_size,
    c=PALETTE["outer"],
    alpha=0.26,
    edgecolor="#FFFFFF",
    linewidth=1,
    zorder=3
)

ax.scatter(
    service["map_x"],
    service["map_y"],
    s=inner_size,
    c=PALETTE["inner"],
    alpha=0.48,
    edgecolor="#FFFFFF",
    linewidth=1,
    zorder=4
)

OUT = Path.cwd() / "figures" / "panels" / "Figure2_a.png"
OUT.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(
    OUT,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.02,
    transparent=True
)

plt.show()

pass


## Panel A1

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"

OUT = Path(
    Path.cwd() / "figures" / "panels" / "Figure2_a1.png"
)

INK = "#263645"
MUTED = "#718096"
LIGHT = "#E8EEF1"

SEASON = {
    "Spring": "#FFFFFF",
    "Summer": "#FFFFFF",
    "Autumn": "#FFFFFF",
    "Winter": "#FFFFFF",
}

REGION = {
    "Northwest": "#9b8f81",
    "North China": "#ebacac",
    "Northeast": "#a9a9ae",
    "East": "#8dbcf6",
    "Central": "#f6e1a5",
    "South China": "#86c183",
    "Southwest": "#e1fb62",
}

CLUSTER = {
    "Stable backbone": "#2D9486",
    "Daylight window": "#D9A12A",
    "Seasonal / episodic": "#8B70AD",
}

FLOW_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "clock_flow",
    ["#F3F6F7", "#C4DCD8", "#62AEA1", "#285B70"]
)

def setup_style() -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
            "font.size": 7.2,
            "axes.titlesize": 8.0,
            "axes.labelsize": 7.0,
            "xtick.labelsize": 6.0,
            "ytick.labelsize": 6.0,
            "figure.facecolor": "none",
            "axes.facecolor": "none",
            "savefig.facecolor": "none",
            "svg.fonttype": "none",
            "pdf.fonttype": 42,
        }
    )

def part(data: pd.DataFrame, name: str) -> pd.DataFrame:
    return data[data["atlas_table"].eq(name)].copy().dropna(axis=1, how="all")

def draw_arc_text(
    ax,
    text: str,
    radius: float,
    theta_center: float = 45.0,
    arc_per_char: float = 7.2,
    min_span: float = 34.0,
    max_span: float = 105.0,
    fontsize: float = 6.3,
    color: str = INK,
    fontweight: str = "bold",
) -> None:
    pass
    text = str(text)
    chars = list(text)

    if len(chars) == 0:
        return

    if len(chars) == 1:
        angles = np.array([theta_center])
    else:
        span = min(max_span, max(min_span, len(chars) * arc_per_char))
        angles = np.linspace(
            theta_center + span / 2,
            theta_center - span / 2,
            len(chars)
        )

    for ch, theta in zip(chars, angles):
        if ch == " ":
            continue

        rad = np.deg2rad(theta)
        x = radius * np.cos(rad)
        y = radius * np.sin(rad)

        ax.text(
            x,
            y,
            ch,
            ha="center",
            va="center",
            fontsize=fontsize,
            color=color,
            fontweight=fontweight,
            rotation=theta - 90,
            rotation_mode="anchor",
        )

def draw_supplier_glyph(
    ax,
    routes: pd.DataFrame,
    target: str,
    imports: float,
    effective: float,
    nominal: int,
    region_map: dict[str, str],
    max_imports: float,
) -> None:
    frame = (
        routes[routes["target_en"].eq(target)]
        .sort_values("flow_twh_it", ascending=False)
        .head(nominal)
        .reset_index(drop=True)
    )

    radius = 0.76 + 0.22 * np.sqrt(imports / max(max_imports, 1e-9))
    inner_radius = radius * 0.48

    total = frame["flow_twh_it"].sum()
    start = 90.0

    for row in frame.itertuples():
        share = row.flow_twh_it / max(total, 1e-12)
        end = start - 360.0 * share
        colour = REGION.get(region_map.get(row.source_en, ""), "#AAB4BD")

        ax.add_patch(
            Wedge(
                (0, 0),
                radius * 0.82,
                end,
                start,
                width=radius * 0.30,
                facecolor=colour,
                edgecolor="white",
                linewidth=0.7,
            )
        )

        start = end

    n = nominal

    for i, row in enumerate(frame.itertuples()):
        theta2 = 90.0 - 360.0 * i / max(n, 1)
        theta1 = 90.0 - 360.0 * (i + 1) / max(n, 1)
        colour = REGION.get(region_map.get(row.source_en, ""), "#AAB4BD")

        ax.add_patch(
            Wedge(
                (0, 0),
                radius,
                theta1,
                theta2,
                width=radius * 0.095,
                facecolor=colour,
                edgecolor="white",
                linewidth=0.55,
            )
        )

    ax.add_patch(
        Circle(
            (0, 0),
            inner_radius,
            facecolor="white",
            edgecolor=LIGHT,
            lw=0.8
        )
    )

    ax.text(
        0,
        0.06,
        f"{effective:.1f}",
        ha="center",
        va="center",
        fontsize=9.0,
        fontweight="bold",
        color=INK,
    )

    ax.text(
        0,
        -0.23,
        "effective",
        ha="center",
        va="center",
        fontsize=4.6,
        color=MUTED
    )

    draw_arc_text(
        ax,
        target,
        radius=radius * 1.18,
        theta_center=45.0,
        arc_per_char=7.2,
        min_span=34.0,
        max_span=105.0,
        fontsize=10,
        color=INK,
        fontweight="bold",
    )

    ax.text(
        0,
        -radius - 0.18,
        f"{imports:.1f} TWh-IT  |  {n} nominal",
        ha="center",
        va="top",
        fontsize=4.9,
        color=MUTED,
    )

    ax.set_xlim(-1.35, 1.55)
    ax.set_ylim(-1.32, 1.45)
    ax.set_aspect("equal")
    ax.set_facecolor("none")
    ax.axis("off")

def plot_panel_h(data: pd.DataFrame) -> Path:
    routes = part(data, "f2_annual_routes")
    redundancy = part(data, "deep_network_target_redundancy")
    roles = part(data, "f2_province_roles")

    routes = routes[routes["source_en"].isin(roles["province_en"])].copy()
    region_map = roles.set_index("province_en")["region"].to_dict()

    selected = redundancy.nlargest(12, "imports_twh_it").reset_index(drop=True)
    max_imports = selected["imports_twh_it"].max()

    fig = plt.figure(figsize=(8.1, 5.05))
    fig.patch.set_alpha(0)

    grid = fig.add_gridspec(
        4,
        4,
        height_ratios=[0.30, 1, 1, 1],
        hspace=0.16,
        wspace=0.05,
        left=0.035,
        right=0.985,
        bottom=0.035,
        top=0.94,
    )

    title_ax = fig.add_subplot(grid[0, :])
    title_ax.set_facecolor("none")
    title_ax.axis("off")

    title_ax.text(
        0,
        0.78,
        'h',
        fontsize=9.2,
        fontweight="bold",
        color=INK,
        ha="left",
        va="center",
    )

    pass

    x = 0.59

    for region, colour in REGION.items():
        title_ax.scatter(
            x,
            0.20,
            s=18,
            color=colour,
            clip_on=False
        )

        title_ax.text(
            x + 0.014,
            0.20,
            region.replace(" China", ""),
            va="center",
            fontsize=4.8,
            color=INK
        )

        x += 0.058 if region != "Inner Mongolia" else 0.08

    title_ax.set_xlim(0, 1)
    title_ax.set_ylim(0, 1)

    for i, row in selected.iterrows():
        ax = fig.add_subplot(grid[1 + i // 4, i % 4])
        ax.set_facecolor("none")

        draw_supplier_glyph(
            ax,
            routes,
            row["target"],
            row["imports_twh_it"],
            row["effective_supplier_count"],
            int(row["supplier_count"]),
            region_map,
            max_imports,
        )

    OUT.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        OUT,
        dpi=300,
        facecolor="none",
        bbox_inches="tight",
        pad_inches=0.02,
        transparent=True
    )

    plt.close(fig)

    return OUT

setup_style()

data = pd.read_feather(DATA)

OUT_PATH = plot_panel_h(data)

from IPython.display import Image, display
display(Image(filename=str(OUT_PATH)))

pass


## Panel B

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

CSV = Path("figure4_routing_backbone_ONE_CSV.csv")
PALETTE = {"grid": "#D8DEE9"}

REGION_ORDER = [
    "Northwest",
    "North China",
    "Northeast",
    "East",
    "Central",
    "South China",
    "Southwest",
]

REGION_BG_COLORS = {
    "Southwest": "#ecfdc025",
    "South China": "#ecfdc025",
    "East": "#e0ebf925",
    "Central": "#faf4e125",
    "North China": "#fce9ea25",
    "Northeast": "#e4e4e625",
    "Northwest": "#e6d9cb25",
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 11,
    "text.color": "#1F2933",
    "figure.dpi": 120,
})

df = load_integrated_table("figure4_routing_backbone_ONE_CSV.csv", low_memory=False)

pb0 = df[df["table"].eq("province_balance")].copy()

mask_hainan = pb0["province_en"].astype(str).str.lower().eq("hainan")

if "province" in pb0.columns:
    mask_hainan = mask_hainan | pb0["province"].astype(str).eq('Hainan')

pb0 = pb0[~mask_hainan].copy()

pb = pb0.copy()

pb["region_order"] = pd.Categorical(
    pb["region"],
    categories=REGION_ORDER,
    ordered=True
)

pb = (
    pb.sort_values(
        ["region_order", "executed_supply_twh_it"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

x_pos = np.arange(len(pb))

line_col = np.where(
    pb["net_output_twh_it"] >= 0,
    "#2D8C76",
    "#B04A4A"
)

fig, ax = plt.subplots(
    figsize=(4.8, 2.2),
    dpi=300,
    facecolor="white"
)

y_max = max(
    float(pb["local_demand_twh_it"].max()),
    float(pb["executed_supply_twh_it"].max())
)
y_top = y_max * 1.04

for region in REGION_ORDER:
    sub_pb = pb[pb["region"].eq(region)]

    if sub_pb.empty:
        continue

    pos = sub_pb.index.to_numpy()

    x_start = min(pos) - 0.5
    x_end = max(pos) + 0.5

    bg_color = REGION_BG_COLORS.get(region, "#F3F4F6")

    ax.axvspan(
        x_start,
        x_end,
        color=bg_color,
        alpha=0.65,
        zorder=0
    )

    ax.axvline(
        x_end,
        color="white",
        lw=1.2,
        zorder=1
    )

for xi, (_, row), col in zip(x_pos, pb.iterrows(), line_col):
    ax.plot(
        [xi, xi],
        [row["local_demand_twh_it"], row["executed_supply_twh_it"]],
        color=col,
        lw=1.2,
        alpha=0.78,
        zorder=2
    )

ax.scatter(
    x_pos,
    pb["local_demand_twh_it"],
    s=28,
    color="#CBD5E1",
    edgecolor="#374151",
    linewidth=0.5,
    label="Own demand",
    zorder=3
)

ax.scatter(
    x_pos,
    pb["executed_supply_twh_it"],
    s=32,
    color=line_col,
    edgecolor="white",
    linewidth=0.6,
    label="Executed supply",
    zorder=3
)

ax.set_xticks(x_pos)
ax.set_xticklabels(
    pb["province_en"],
    fontsize=7.5,
    rotation=90,
    ha="center"
)

ax.set_ylabel("")

ax.tick_params(axis="y", labelsize=10.5)

ax.grid(
    True,
    axis="y",
    color=PALETTE["grid"],
    lw=0.55,
    alpha=0.58,
    zorder=1
)

ax.set_ylim(0, y_top * 1.10)
ax.set_xlim(-0.5, len(pb) - 0.5)

ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


## Panel CDE

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

data = load_integrated_table("figure35_hourly_dispatch_carrier_design_v2_ONE_CSV.csv")

TECH_COLORS = {"Solar": "#D8A21B", "Wind": "#2E6E9E", "Hydro": "#2C8C7C"}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 18.0,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.72,
    "axes.edgecolor": "#2F3640",
    "legend.frameon": False,
    "figure.dpi": 120
})

def get_table(name):
    return data[data["table"].eq(name)].copy()

coords = get_table("map_coords")
boundary = get_table("china_boundary_segments")
ternary_routes = get_table("panel_a_ternary")
routes = pd.concat([
    ternary_routes.assign(
        carrier=label,
        carrier_weighted_twh_it=pd.to_numeric(ternary_routes[value_col], errors="coerce"),
    ).loc[lambda x: x["carrier_weighted_twh_it"] > 0].nlargest(
        8, "carrier_weighted_twh_it"
    ).copy()
    for label, value_col in [
        ("Solar", "solar_weighted_twh_it"),
        ("Wind", "wind_weighted_twh_it"),
        ("Hydro", "hydro_weighted_twh_it"),
    ]
], ignore_index=True)

xy = coords.set_index("province")[["map_x", "map_y"]].to_dict("index")

def draw_boundary(ax):
    pass
    for _, seg in boundary.groupby("segment_id"):
        seg = seg.sort_values("point_order")
        ax.plot(seg["map_x"], seg["map_y"], color="#9EA7B300", lw=0.50, zorder=1)

def draw_arrow(ax, p0, p1, width, color, alpha, rad=0.16):
    ax.add_patch(FancyArrowPatch(
        p0, p1, connectionstyle=f"arc3,rad={rad}", arrowstyle="simple",
        mutation_scale=6.5 + width * 1.5,
        lw=0.0,
        linewidth=0.0,
        edgecolor="none",
        color=color, alpha=alpha,
        shrinkA=4.0, shrinkB=4.0, zorder=4
    ))

def draw_single_carrier_map(tech_name, title_text, filename):
    pass
    fig, ax = plt.subplots(figsize=(4.8, 3.5), dpi=300)

    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    draw_boundary(ax)

    sub = routes[routes["carrier"].eq(tech_name)].copy().sort_values("carrier_weighted_twh_it")
    max_val = max(sub["carrier_weighted_twh_it"].max(), 1e-9)

    for _, row in sub.iterrows():
        if row["source"] not in xy or row["target"] not in xy:
            continue
        width = 1.6875 + 13.5 * row["carrier_weighted_twh_it"] / max_val
        rad_val = 0.13 if tech_name != "Hydro" else -0.10

        draw_arrow(
            ax,
            (xy[row["source"]]["map_x"], xy[row["source"]]["map_y"]),
            (xy[row["target"]]["map_x"], xy[row["target"]]["map_y"]),
            width, TECH_COLORS[tech_name], 0.72, rad=rad_val
        )

    ax.scatter(
        coords["map_x"], coords["map_y"],
        s=15, color="#6B7280", alpha=0.35,
        edgecolor="white", linewidth=0.3, zorder=3
    )

    for name in ["Xinjiang", "Gansu", "Sichuan", "Yunnan", "Jiangsu", "Guangdong", "Shanghai"]:
        hit = coords[coords["province_en"].eq(name)]
        if not hit.empty:
            ax.text(
                hit["map_x"].iloc[0] + 65, hit["map_y"].iloc[0] + 42, name,
                fontsize=12, color="#26323F", weight="semibold", zorder=5,
                bbox=dict(boxstyle="round,pad=0.05", fc="white", ec="none", alpha=0.1)
            )

    ax.set_aspect("equal")

    ax.set_xlim(coords["map_x"].min() - 1300, coords["map_x"].max() + 650)
    ax.set_ylim(coords["map_y"].min() - 550, coords["map_y"].max() + 550)
    ax.axis("off")

    plt.savefig(filename, dpi=300, transparent=True, bbox_inches="tight")
    plt.show()

configs = [
    {
        "tech": "Solar",
        "title": "e | Solar-window corridors",
        "file": str(Path.cwd() / "figures" / "panels" / "Figure2_c.png")
    },
    {
        "tech": "Wind",
        "title": "f | Dark-hour wind corridors",
        "file": str(Path.cwd() / "figures" / "panels" / "Figure2_d.png")
    },
    {
        "tech": "Hydro",
        "title": "g | Dark-hour hydro corridors",
        "file": str(Path.cwd() / "figures" / "panels" / "Figure2_e.png")
    }
]

for cfg in configs:
    draw_single_carrier_map(cfg["tech"], cfg["title"], cfg["file"])


## Panel F

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

data = load_integrated_table("figure35_hourly_dispatch_carrier_design_v2_ONE_CSV.csv")

TECH_COLORS = {
    "Solar": "#D8A21B",
    "Wind":  "#2E6E9E",
    "Hydro": "#2C8C7C"
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8.0,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.72,
    "axes.edgecolor": "#2F3640",
    "legend.frameon": False,
    "figure.dpi": 120
})

def get_table(name):
    return data[data["table"].eq(name)].copy()

df = get_table("panel_a_ternary").copy()

fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=300, facecolor="white")
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

tri = np.array([
    [0, 0],
    [1, 0],
    [0.5, np.sqrt(3) / 2],
    [0, 0]
])

ax.plot(tri[:, 0], tri[:, 1], color="#26323F", lw=1.0)

for frac in [0.25, 0.5, 0.75]:
    ax.plot(
        [frac, 0.5 + frac / 2],
        [0, np.sqrt(3) / 2 * (1 - frac)],
        color="#D8DEE6", lw=0.45
    )
    ax.plot(
        [1 - frac, 0.5 - frac / 2],
        [0, np.sqrt(3) / 2 * (1 - frac)],
        color="#D8DEE6", lw=0.45
    )
    ax.plot(
        [frac / 2, 1 - frac / 2],
        [np.sqrt(3) / 2 * frac, np.sqrt(3) / 2 * frac],
        color="#D8DEE6", lw=0.45
    )

max_val = df["carrier_total_twh_it"].max() or 1e-9
df["pt_size"] = 70 + 280 * df["carrier_total_twh_it"] / max_val

ax.scatter(
    df["ternary_x"],
    df["ternary_y"],
    s=df["pt_size"],
    c=[TECH_COLORS[d] for d in df["dominant_carrier"]],
    alpha=0.72,
    edgecolor="white",
    linewidth=0.6,
    zorder=3
)

N_LABEL_PER_CARRIER = 5

label_df = pd.concat([
    df[df["dominant_carrier"].eq(k)]
    .sort_values("carrier_total_twh_it", ascending=False)
    .head(N_LABEL_PER_CARRIER)
    for k in ["Solar", "Wind", "Hydro"]
]).drop_duplicates("corridor").copy()

labels_to_draw = []

for _, row in label_df.iterrows():
    name = row["corridor"].replace("Inner Mongolia", "Inner M.")
    x_orig = row["ternary_x"]
    y_orig = row["ternary_y"]

    x_offset = 0.032 + 0.0023 * np.sqrt(row["pt_size"])

    labels_to_draw.append({
        "name": name,
        "x_orig": x_orig,
        "x_target": x_orig + x_offset,
        "y_orig": y_orig,
        "y_cur": y_orig
    })

labels_to_draw.sort(key=lambda x: x["y_orig"])

min_text_gap = 0.036

for i in range(1, len(labels_to_draw)):
    if labels_to_draw[i]["y_cur"] - labels_to_draw[i - 1]["y_cur"] < min_text_gap:
        labels_to_draw[i]["y_cur"] = labels_to_draw[i - 1]["y_cur"] + min_text_gap

for i in range(len(labels_to_draw) - 2, -1, -1):
    if labels_to_draw[i + 1]["y_cur"] - labels_to_draw[i]["y_cur"] < min_text_gap:
        labels_to_draw[i]["y_cur"] = labels_to_draw[i + 1]["y_cur"] - min_text_gap

for item in labels_to_draw:
    item["y_cur"] = min(max(item["y_cur"], -0.01), 0.91)

for i, item in enumerate(labels_to_draw):
    stagger = (i % 3) * 0.018
    item["x_target"] = item["x_target"] + stagger

for item in labels_to_draw:
    item["x_target"] = min(item["x_target"], 1.10)

for item in labels_to_draw:
    ax.text(
        item["x_target"],
        item["y_cur"],
        item["name"],
        fontsize=6.2,
        va="center",
        ha="left",
        weight="semibold",
        color="#1F2933",
        zorder=4,
        bbox=dict(
            boxstyle="round,pad=0.06",
            fc="white",
            ec="none",
            alpha=0.80
        )
    )

ax.text(
    -0.04, -0.04,
    "Solar",
    ha="right", va="top",
    color=TECH_COLORS["Solar"],
    fontweight="bold",
    fontsize=9.5
)

ax.text(
    1.04, -0.04,
    "Wind",
    ha="left", va="top",
    color=TECH_COLORS["Wind"],
    fontweight="bold",
    fontsize=9.5
)

ax.text(
    0.5, np.sqrt(3) / 2 + 0.04,
    "Hydro",
    ha="center", va="bottom",
    color=TECH_COLORS["Hydro"],
    fontweight="bold",
    fontsize=9.5
)

ax.set_aspect("equal")
ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-0.12, 0.98)
ax.axis("off")

plt.show()


## Panel G

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.lines import Line2D

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "main_figure3_a_swing_emphasis_old_style.png"

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 11,
    'text.color': '#1F2933',
    'figure.dpi': 120,
})

data = pd.read_feather(DATA)
all_routes = data[data['atlas_table'].eq('f3_seasonal_all_corridors')].copy().dropna(axis=1, how='all')
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
all_routes[season_order + ['annual']] = all_routes[season_order + ['annual']].apply(pd.to_numeric)

backbone = all_routes.nlargest(5, 'annual').copy()
remaining = all_routes[~all_routes['corridor'].isin(backbone['corridor'])].copy()
mean_flow = remaining[season_order].mean(axis=1)
remaining['seasonal_cv'] = remaining[season_order].std(axis=1) / mean_flow.replace(0, np.nan)
remaining['swing_score'] = remaining['seasonal_cv'].fillna(0) * np.sqrt(remaining['annual'].clip(lower=0))
swing = remaining[remaining['annual'].ge(1.0)].nlargest(7, 'swing_score').copy()
selected = pd.concat([backbone, swing], ignore_index=True).drop_duplicates('corridor')

unselected = all_routes[~all_routes['corridor'].isin(selected['corridor'])].copy()
northwest_sources = {'Xinjiang', 'Gansu', 'Qinghai', 'Ningxia', 'Shaanxi'}
def residual_family(source):
    if source == 'Yunnan':
        return 'Other Yunnan corridors'
    if source == 'Sichuan':
        return 'Other Sichuan corridors'
    if source in northwest_sources:
        return 'Other Northwest corridors'
    return 'Other minor corridors'

unselected['corridor'] = unselected['source'].map(residual_family)
bundles = unselected.groupby('corridor', as_index=False)[season_order + ['annual']].sum()
bundle_source = {
    'Other Yunnan corridors': 'Yunnan',
    'Other Sichuan corridors': 'Sichuan',
    'Other Northwest corridors': 'Gansu',
    'Other minor corridors': 'Other',
}
bundles['source'] = bundles['corridor'].map(bundle_source)
bundles['target'] = ''
main = pd.concat([selected, bundles], ignore_index=True).set_index('corridor')

season_colors = {'Spring': '#73C49E', 'Summer': '#E1B33E', 'Autumn': '#D4775D', 'Winter': '#6B8CC7'}
source_colors = {
    'Yunnan': '#2F8F83', 'Sichuan': '#4E9A67', 'Inner M.': '#5D7EA6',
    'Inner Mongolia': '#5D7EA6', 'Xinjiang': '#B88740', 'Gansu': '#C49A49',
    'Guangxi': '#66A9A0', 'Ningxia': '#A27BB8', 'Shanxi': '#8A8F5A',
    'Qinghai': '#7AA6C2', 'Other': '#97999C',
}

normal_corridors = sorted([c for c in main.index if c != 'Other minor corridors'])
order = ['Other minor corridors'] + sorted(normal_corridors, reverse=True)

fig, ax = plt.subplots(figsize=(8, 3.9), dpi=300, facecolor='white')
ax.set_xlim(-0.18, 3.30)
ax.set_ylim(-0.08, 1.08)
ax.axis('off')

xpos = {s: i for i, s in enumerate(season_order)}
national_totals = {s: float(all_routes[s].sum()) for s in season_order}
max_total = max(max(national_totals.values()), 1e-9)
intervals = {s: {} for s in season_order}

for season in season_order:
    total = national_totals[season]
    y0 = 0.5 - 0.82 * (total / max_total) / 2
    for corridor in order:
        height = 0.82 * float(main.loc[corridor, season]) / max_total
        intervals[season][corridor] = (y0, y0 + height)
        y0 += height

for corridor in order:
    source = main.loc[corridor, 'source'] if main.loc[corridor, 'source'] in source_colors else 'Other'
    colour = '#B9C0C8' if corridor == 'Other minor corridors' else source_colors[source]
    for left, right in zip(season_order[:-1], season_order[1:]):
        y00, y01 = intervals[left][corridor]
        y10, y11 = intervals[right][corridor]
        if (y01 - y00) <= 0 and (y11 - y10) <= 0:
            continue
        x0, x1 = xpos[left] + 0.08, xpos[right] - 0.08
        verts = [
            (x0, y00), (x0 + 0.35, y00), (x1 - 0.35, y10), (x1, y10),
            (x1, y11), (x1 - 0.35, y11), (x0 + 0.35, y01), (x0, y01), (x0, y00),
        ]
        codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
                 MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY]
        ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=colour, edgecolor='white',
                               lw=0.25, alpha=0.40 if corridor != 'Other minor corridors' else 0.18, zorder=1))

for season in season_order:
    x = xpos[season]
    ax.text(x, 1.025, season, ha='center', va='bottom', fontsize=9,
            fontweight='bold', color=season_colors[season])
    ax.text(x, -0.01, f'{national_totals[season]:.1f} TWh-IT', ha='center', va='top',
            fontsize=7, color='#444')
    for corridor in order:
        y0, y1 = intervals[season][corridor]
        source = main.loc[corridor, 'source'] if main.loc[corridor, 'source'] in source_colors else 'Other'
        colour = '#B9C0C8' if corridor == 'Other minor corridors' else source_colors[source]
        ax.add_patch(Rectangle((x - 0.075, y0), 0.15, y1 - y0, fc=colour, ec='white',
                               lw=0.25, alpha=0.88 if corridor != 'Other minor corridors' else 0.48, zorder=3))

label_data = []
for corridor in order:
    ys0, ys1 = intervals['Summer'][corridor]
    ya0, ya1 = intervals['Autumn'][corridor]
    if (ys1 - ys0) > 0 or (ya1 - ya0) > 0:
        centre = ((ys0 + ys1) / 2 + (ya0 + ya1) / 2) / 2
        label_data.append({'name': corridor, 'y_orig': centre, 'y_cur': centre})
label_data.sort(key=lambda item: item['y_orig'])
minimum_gap = 0.024
for i in range(1, len(label_data)):
    if label_data[i]['y_cur'] - label_data[i - 1]['y_cur'] < minimum_gap:
        label_data[i]['y_cur'] = label_data[i - 1]['y_cur'] + minimum_gap
for i in range(len(label_data) - 2, -1, -1):
    label_data[i + 1]['y_cur'] = min(label_data[i + 1]['y_cur'], 0.99)
    if label_data[i + 1]['y_cur'] - label_data[i]['y_cur'] < minimum_gap:
        label_data[i]['y_cur'] = label_data[i + 1]['y_cur'] - minimum_gap
for item in label_data:
    ax.text(1.5, item['y_cur'], item['name'], ha='center', va='center', fontsize=5.8,
            color='#1F2933', weight='semibold',
            bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.55), zorder=4)

ax.text(-0.16, 0.02,
        'Band colour = executing/resource province family; band thickness = seasonal gross routed compute',
        fontsize=6.8, color='#555', ha='left')
legend_order = ['Yunnan', 'Sichuan', 'Inner Mongolia', 'Xinjiang', 'Gansu', 'Qinghai', 'Other']
handles = [Line2D([0], [0], color=source_colors[name], lw=5,
                  label='Inner M.' if name == 'Inner Mongolia' else name) for name in legend_order]
ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.50, -0.075),
          ncol=7, frameon=False, fontsize=6.3)

OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=300, facecolor='white', bbox_inches='tight')
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Panel H

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8.0,
    "text.color": "#1F2933",
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

SEASON_COLORS = {
    "Spring": "#61B58B",
    "Summer": "#D9A21B",
    "Autumn": "#C76C4A",
    "Winter": "#6E8FD3"
}

REGION_COLORS = {
    "East": "#4E79A7",
    "South China": "#59A14F",
    "Central": "#F28E2B",
    "North China": "#E15759",
    "Southwest": "#8CD17D",
    "Northwest": "#B07AA1",
    "Northeast": "#9C755F"
}

REGION_LABELS = {
    "Southwest": "SW",
    "Northwest": "NW",
    "North China": "North",
    "Northeast": "NE",
    "East": "East",
    "Central": "Central",
    "South China": "South"
}

NODE_ORDER = [
    "Southwest",
    "Northwest",
    "North China",
    "Northeast",
    "East",
    "Central",
    "South China"
]

def load_network_data():
    pass
    raw = load_integrated_table("figure33_seasonal_network_structure_supplement_ONE_CSV.csv")
    df = raw[raw["table"] == "seasonal_corridors"].dropna(axis=1, how="all").copy()
    df = df.drop(columns=["table"])

    for col in df.columns:
        if col not in ["source_region", "target_region", "season", "corridor", "source_family"]:
            df[col] = pd.to_numeric(df[col], errors="ignore")

    return df

def draw_single_season_jupyter(season, seasonal_routes):
    pass

    fig, ax = plt.subplots(figsize=(2.8, 2.8), dpi=300, facecolor="white")
    fig.patch.set_facecolor("white")
    fig.patch.set_alpha(1.0)
    ax.set_facecolor("white")
    ax.patch.set_alpha(1.0)

    ax.set_aspect("equal")
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.set_axis_off()

    angles = np.linspace(
        np.pi / 2,
        np.pi / 2 - 2 * np.pi,
        len(NODE_ORDER),
        endpoint=False
    )
    pos = {
        reg: np.array([np.cos(a), np.sin(a)])
        for reg, a in zip(NODE_ORDER, angles)
    }

    max_flow = seasonal_routes.groupby(
        ["season", "source_region", "target_region"]
    )["flow_twh_it"].sum().max()
    max_flow = max(float(max_flow), 1e-9)

    season_df = seasonal_routes[seasonal_routes["season"] == season]
    edges = season_df.groupby(
        ["source_region", "target_region"],
        as_index=False
    )["flow_twh_it"].sum()

    edges = edges[
        (edges["source_region"] != edges["target_region"]) &
        (edges["flow_twh_it"] > 0)
    ]

    top_edges = edges.sort_values("flow_twh_it", ascending=False)

    for _, row in top_edges.iloc[::-1].iterrows():
        src, dst = row["source_region"], row["target_region"]

        if src not in pos or dst not in pos:
            continue

        strength = float(row["flow_twh_it"]) / max_flow

        arrow = FancyArrowPatch(
            pos[src] * 0.86,
            pos[dst] * 0.86,
            connectionstyle="arc3,rad=0.26",
            arrowstyle="-|>",
            mutation_scale=6 + 9 * strength,
            lw=0.75 + 5.8 * strength,
            color=REGION_COLORS.get(src, "#9EA7B3"),
            alpha=0.20 + 0.52 * strength,
            zorder=1
        )

        ax.add_patch(arrow)

    gross = {}

    for reg in NODE_ORDER:
        in_flow = season_df[season_df["target_region"] == reg]["flow_twh_it"].sum()
        out_flow = season_df[season_df["source_region"] == reg]["flow_twh_it"].sum()
        gross[reg] = float(in_flow + out_flow)

    max_gross = max(gross.values()) or 1

    for reg in NODE_ORDER:
        xy = pos[reg]
        node_size = 70 + 260 * np.sqrt(gross[reg] / max_gross)

        ax.scatter(
            [xy[0]],
            [xy[1]],
            s=node_size,
            color=REGION_COLORS.get(reg, "#9EA7B3"),
            edgecolor="white",
            linewidth=0.8,
            zorder=3
        )

        ax.text(
            xy[0] * 1.19,
            xy[1] * 1.19,
            REGION_LABELS.get(reg, reg),
            fontsize=6.5,
            ha="center",
            va="center"
        )

    total_flow = season_df["flow_twh_it"].sum()

    ax.text(
        -1.25,
        1.22,
        season,
        color=SEASON_COLORS[season],
        fontsize=9.5,
        fontweight="bold",
        ha="left",
        va="top"
    )

    ax.text(
        -1.25,
        1.04,
        f"{total_flow:.1f} TWh-IT",
        color="#374151",
        fontsize=6.8,
        ha="left",
        va="top"
    )

    plt.show()

seasonal_corridors_df = load_network_data()

for target_season in ["Spring", "Summer", "Autumn", "Winter"]:
    draw_single_season_jupyter(target_season, seasonal_corridors_df)


## Panel I

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

CSV = Path("figure35_hourly_dispatch_carrier_design_v2_ONE_CSV.csv")
if not CSV.exists():
    CSV = Path("source_data") / "figure35_hourly_dispatch_carrier_design_v2_ONE_CSV.csv"
data = load_integrated_table('figure35_hourly_dispatch_carrier_design_v2_ONE_CSV.csv', low_memory=False)

TECH_COLORS = {"Solar": "#D8A21B", "Wind": "#2E6E9E", "Hydro": "#2C8C7C"}
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 10,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.72,
    "axes.edgecolor": "#2F3640",
    "legend.frameon": False,
})

def get_table(name):
    return data[data["table"].eq(name)].copy()

profiles = get_table("panel_c_corridor_fingerprints")
fig, ax = plt.subplots(figsize=(7.0, 4),dpi=300)
order = profiles.groupby("corridor")["raw_gwh_it_h"].sum().sort_values(ascending=False).index.tolist()
offsets = np.arange(len(order))[::-1]
for offset, corridor in zip(offsets, order):
    sub = profiles[profiles["corridor"].eq(corridor)].sort_values("hour")
    total = sub[["solar_weighted_gwh_it_h", "wind_weighted_gwh_it_h", "hydro_weighted_gwh_it_h"]].sum(axis=1)
    if total.max() <= 0:
        continue
    y0 = np.full(len(sub), offset, dtype=float)
    bottom = y0.copy()
    scale = 0.72 / total.max()
    for col, carrier_name in [("solar_weighted_gwh_it_h", "Solar"), ("wind_weighted_gwh_it_h", "Wind"), ("hydro_weighted_gwh_it_h", "Hydro")]:
        vals = sub[col].to_numpy() * scale
        ax.fill_between(sub["hour"], bottom, bottom + vals, color=TECH_COLORS[carrier_name], alpha=0.55, linewidth=0)
        bottom += vals
    ax.plot(sub["hour"], y0 + total.to_numpy() * scale, color="#26323F", lw=0.55, alpha=0.65)

    ax.text(
        2.0, offset + 0.35, corridor.replace("Inner Mongolia", "Inner M."),
        fontsize=10, va="center", ha="left", weight="semibold", zorder=5,
        bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.65)
    )

ax.axvspan(10, 15, color="#D8A21B", alpha=0.08, lw=0)
ax.axvspan(18, 23, color="#19324F", alpha=0.06, lw=0)
ax.axvspan(0, 5, color="#19324F", alpha=0.06, lw=0)
ax.set_xlim(0, 24.0)
ax.set_xticks([0, 6, 10, 15, 18, 23])
ax.set_yticks([])
ax.tick_params(axis="x", labelsize=10)
ax.set_xlabel("Clock hour")

ax.grid(axis="x", color="#E4E9EF", lw=0.55)
ax.legend(handles=[mpl.patches.Patch(facecolor=c, alpha=0.55, label=k) for k, c in TECH_COLORS.items()], loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=3, fontsize=9, frameon=False, handlelength=1.4, columnspacing=1.0, borderaxespad=0.2)
plt.show()


## Panel J

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "national_hourly_shift_annual_mean.png"

INK = '#27364A'
MUTED = '#8B95A1'
GOLD = '#D89C45'
BLUE = '#5B7DB8'
GREEN = '#357B43'
AMBER = '#C58900'
GRID = '#DCE4EA'

AXIS_BLACK = "#000000"

AXIS_LW = 1.05
TICK_LW = 0.95
TICK_LEN = 4.0

LABEL_FONT = 13
TICK_FONT = 10
LEGEND_FONT = 8.0

mpl.rcdefaults()
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 10,

    "axes.spines.top": False,
    "axes.linewidth": AXIS_LW,
    "axes.edgecolor": AXIS_BLACK,

    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,

    "legend.frameon": True,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

data = pd.read_feather(DATA)

frame = data[
    data["atlas_table"].eq("f3_national_hourly_shift_periods")
    & data["period"].eq("Annual mean")
].copy().sort_values("hour")

hours = frame["hour"].astype(int).to_numpy()
shift = frame["shifted_workload_gwh_it_h"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7.0, 4.0), dpi=300)

ax.bar(
    hours,
    shift,
    width=0.78,
    color=np.where(shift >= 0, GOLD, BLUE),
    alpha=0.75,
    zorder=2
)

ax.axhline(0, color=MUTED, lw=0.70)
ax.grid(axis="both", color=GRID, lw=0.55, alpha=0.75)
ax.set_axisbelow(True)

ax.set_xlim(-0.6, 23.6)
ax.set_xticks([0, 6, 12, 18, 23])

ax.set_xlabel("Clock hour", fontsize=LABEL_FONT)
ax.set_ylabel("Shifted IT workload (GWh-IT h$^{-1}$)", fontsize=LABEL_FONT)

ax2 = ax.twinx()

ax2.plot(
    hours,
    frame["demand_static_gwh_h"],
    color=MUTED,
    lw=1.7,
    ls="--",
    label="Static IT workload"
)

ax2.plot(
    hours,
    frame["demand_shifted_gwh_h"],
    color=INK,
    lw=2.1,
    label="After temporal shifting"
)

ax2.plot(
    hours,
    frame["clean_power_used_gwh_h"],
    color=GREEN,
    lw=1.8,
    label="Clean power used"
)

ax2.plot(
    hours,
    frame["routed_compute_gwh_it_h"],
    color=AMBER,
    lw=1.6,
    ls=":",
    label="Routed compute"
)

ax2.set_ylabel("National annual-mean energy (GWh h$^{-1}$)", fontsize=LABEL_FONT)

ax.axvline(
    int(frame["peak_hour"].iloc[0]),
    color=INK,
    lw=0.9,
    ls=":"
)

ax.axvline(
    int(frame["evening_hour"].iloc[0]),
    color=AMBER,
    lw=0.9,
    ls=":"
)

for side in ["left", "bottom"]:
    ax.spines[side].set_linewidth(AXIS_LW)
    ax.spines[side].set_color(AXIS_BLACK)

ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

ax2.spines["right"].set_linewidth(AXIS_LW)
ax2.spines["right"].set_color(AXIS_BLACK)
ax2.spines["top"].set_visible(False)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=TICK_FONT,
    width=TICK_LW,
    length=TICK_LEN,
    colors=AXIS_BLACK
)

ax2.tick_params(
    axis="y",
    which="major",
    labelsize=TICK_FONT,
    width=TICK_LW,
    length=TICK_LEN,
    colors=AXIS_BLACK
)

ax.xaxis.label.set_color(AXIS_BLACK)
ax.yaxis.label.set_color(AXIS_BLACK)
ax2.yaxis.label.set_color(AXIS_BLACK)

handles = [
    Patch(facecolor=GOLD, alpha=0.75, label="Shifted into hour"),
    Patch(facecolor=BLUE, alpha=0.75, label="Shifted out of hour"),

    mpl.lines.Line2D(
        [0], [0],
        color=MUTED,
        lw=1.7,
        ls="--",
        label="Static IT workload"
    ),

    mpl.lines.Line2D(
        [0], [0],
        color=INK,
        lw=2.1,
        label="After temporal shifting"
    ),

    mpl.lines.Line2D(
        [0], [0],
        color=GREEN,
        lw=1.8,
        label="Clean power used"
    ),

    mpl.lines.Line2D(
        [0], [0],
        color=AMBER,
        lw=1.6,
        ls=":",
        label="Routed compute"
    ),
]

leg = ax.legend(
    handles=handles,
    loc="upper left",
    ncol=2,
    fontsize=LEGEND_FONT,
    framealpha=0.92,
    handlelength=1.8,
    columnspacing=1.0,
    handletextpad=0.5,
    borderpad=0.45,
    labelspacing=0.35
)

leg.get_frame().set_linewidth(0.35)
leg.get_frame().set_edgecolor("#D8DEE9")

fig.tight_layout()

OUT.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(
    OUT,
    dpi=300,
    facecolor="white",
    bbox_inches="tight"
)

plt.close(fig)

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "province_hourly_dispatch_spring.png"

INK = '#27364A'
MUTED = '#8B95A1'
GOLD = '#D89C45'
BLUE = '#5B7DB8'
GREEN = '#357B43'
AMBER = '#C58900'
GRID = '#DCE4EA'
SPINE = '#5F6873'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 7.4,
    'axes.linewidth': 0.8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

data = pd.read_feather(
    DATA,
    columns=[
        'atlas_table', 'province_en', 'season', 'hour',
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h', 'peak_hour', 'evening_hour',
    ],
)
all_profiles = data[data['atlas_table'].eq('f3_province_season_hourly_shift')].copy()
profiles = all_profiles[all_profiles['season'].eq('Spring')].copy()
provinces = sorted(profiles['province_en'].dropna().unique())

ncols = 4
nrows = int(np.ceil((len(provinces) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12.2, 16.0), sharex=True)
axes = np.asarray(axes).ravel()

for panel_index, (ax, province) in enumerate(zip(axes, provinces)):
    frame = profiles[profiles['province_en'].eq(province)].sort_values('hour')
    hours = frame['hour'].to_numpy(dtype=int)
    absolute = frame[[
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h',
    ]].to_numpy(dtype=float)
    panel_scale = max(float(np.max(np.abs(absolute))), 1e-9)
    shift = frame['shifted_workload_gwh_it_h'].to_numpy(dtype=float) / panel_scale
    ax.bar(hours, shift, width=0.78, color=np.where(shift >= 0, GOLD, BLUE),
           alpha=0.76, edgecolor='none', zorder=2)
    ax.axhline(0, color=MUTED, lw=0.55)
    ax.set_xlim(-0.6, 23.6)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.grid(axis='both', color=GRID, lw=0.42, alpha=0.60)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=6.3, length=2.2, pad=1.4)
    ax.text(0.025, 0.955, province, transform=ax.transAxes, ha='left', va='top',
            fontsize=7.7, fontweight='bold', color=INK, zorder=8)
    ax.text(0.975, 0.955, f'peak={panel_scale:.2f} GWh h$^{-1}$',
            transform=ax.transAxes, ha='right', va='top', fontsize=5.8,
            color=MUTED, zorder=8)
    ax.plot(hours, frame['demand_static_gwh_h'] / panel_scale,
            color=MUTED, lw=1.05, ls='--')
    ax.plot(hours, frame['demand_shifted_gwh_h'] / panel_scale,
            color=INK, lw=1.30)
    ax.plot(hours, frame['clean_power_used_gwh_h'] / panel_scale,
            color=GREEN, lw=1.15)
    ax.plot(hours, frame['routed_compute_gwh_it_h'] / panel_scale,
            color=AMBER, lw=1.05, ls=':')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(SPINE)
        spine.set_linewidth(0.80)

legend_ax = axes[len(provinces)]
legend_ax.axis('off')
legend_ax.legend(handles=[
    Patch(facecolor=GOLD, alpha=0.76, label='Shifted into hour'),
    Patch(facecolor=BLUE, alpha=0.76, label='Shifted out of hour'),
    mpl.lines.Line2D([0], [0], color=MUTED, lw=1.2, ls='--', label='Static IT workload'),
    mpl.lines.Line2D([0], [0], color=INK, lw=1.4, label='After temporal shifting'),
    mpl.lines.Line2D([0], [0], color=GREEN, lw=1.3, label='Clean power used'),
    mpl.lines.Line2D([0], [0], color=AMBER, lw=1.2, ls=':', label='Routed compute'),
], loc='center', frameon=False, fontsize=8.1, ncol=1)
for ax in axes[len(provinces) + 1:]:
    ax.set_visible(False)

pass
fig.supxlabel('Clock hour', fontsize=9.0, y=0.006)
fig.supylabel('Normalized value (panel peak = 1)', fontsize=9.0, x=0.006)
fig.tight_layout(rect=(0.025, 0.022, 0.998, 0.982), h_pad=0.82, w_pad=0.65)
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=300, facecolor='white', bbox_inches='tight')
plt.close(fig)

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "province_hourly_dispatch_summer.png"

INK = '#27364A'
MUTED = '#8B95A1'
GOLD = '#D89C45'
BLUE = '#5B7DB8'
GREEN = '#357B43'
AMBER = '#C58900'
GRID = '#DCE4EA'
SPINE = '#5F6873'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 7.4,
    'axes.linewidth': 0.8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

data = pd.read_feather(
    DATA,
    columns=[
        'atlas_table', 'province_en', 'season', 'hour',
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h', 'peak_hour', 'evening_hour',
    ],
)
all_profiles = data[data['atlas_table'].eq('f3_province_season_hourly_shift')].copy()
profiles = all_profiles[all_profiles['season'].eq('Summer')].copy()
provinces = sorted(profiles['province_en'].dropna().unique())

ncols = 4
nrows = int(np.ceil((len(provinces) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12.2, 16.0), sharex=True)
axes = np.asarray(axes).ravel()

for panel_index, (ax, province) in enumerate(zip(axes, provinces)):
    frame = profiles[profiles['province_en'].eq(province)].sort_values('hour')
    hours = frame['hour'].to_numpy(dtype=int)
    absolute = frame[[
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h',
    ]].to_numpy(dtype=float)
    panel_scale = max(float(np.max(np.abs(absolute))), 1e-9)
    shift = frame['shifted_workload_gwh_it_h'].to_numpy(dtype=float) / panel_scale
    ax.bar(hours, shift, width=0.78, color=np.where(shift >= 0, GOLD, BLUE),
           alpha=0.76, edgecolor='none', zorder=2)
    ax.axhline(0, color=MUTED, lw=0.55)
    ax.set_xlim(-0.6, 23.6)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.grid(axis='both', color=GRID, lw=0.42, alpha=0.60)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=6.3, length=2.2, pad=1.4)
    ax.text(0.025, 0.955, province, transform=ax.transAxes, ha='left', va='top',
            fontsize=7.7, fontweight='bold', color=INK, zorder=8)
    ax.text(0.975, 0.955, f'peak={panel_scale:.2f} GWh h$^{-1}$',
            transform=ax.transAxes, ha='right', va='top', fontsize=5.8,
            color=MUTED, zorder=8)
    ax.plot(hours, frame['demand_static_gwh_h'] / panel_scale,
            color=MUTED, lw=1.05, ls='--')
    ax.plot(hours, frame['demand_shifted_gwh_h'] / panel_scale,
            color=INK, lw=1.30)
    ax.plot(hours, frame['clean_power_used_gwh_h'] / panel_scale,
            color=GREEN, lw=1.15)
    ax.plot(hours, frame['routed_compute_gwh_it_h'] / panel_scale,
            color=AMBER, lw=1.05, ls=':')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(SPINE)
        spine.set_linewidth(0.80)

legend_ax = axes[len(provinces)]
legend_ax.axis('off')
legend_ax.legend(handles=[
    Patch(facecolor=GOLD, alpha=0.76, label='Shifted into hour'),
    Patch(facecolor=BLUE, alpha=0.76, label='Shifted out of hour'),
    mpl.lines.Line2D([0], [0], color=MUTED, lw=1.2, ls='--', label='Static IT workload'),
    mpl.lines.Line2D([0], [0], color=INK, lw=1.4, label='After temporal shifting'),
    mpl.lines.Line2D([0], [0], color=GREEN, lw=1.3, label='Clean power used'),
    mpl.lines.Line2D([0], [0], color=AMBER, lw=1.2, ls=':', label='Routed compute'),
], loc='center', frameon=False, fontsize=8.1, ncol=1)
for ax in axes[len(provinces) + 1:]:
    ax.set_visible(False)

pass
fig.supxlabel('Clock hour', fontsize=9.0, y=0.006)
fig.supylabel('Normalized value (panel peak = 1)', fontsize=9.0, x=0.006)
fig.tight_layout(rect=(0.025, 0.022, 0.998, 0.982), h_pad=0.82, w_pad=0.65)
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=300, facecolor='white', bbox_inches='tight')
plt.close(fig)

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "province_hourly_dispatch_autumn.png"

INK = '#27364A'
MUTED = '#8B95A1'
GOLD = '#D89C45'
BLUE = '#5B7DB8'
GREEN = '#357B43'
AMBER = '#C58900'
GRID = '#DCE4EA'
SPINE = '#5F6873'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 7.4,
    'axes.linewidth': 0.8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

data = pd.read_feather(
    DATA,
    columns=[
        'atlas_table', 'province_en', 'season', 'hour',
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h', 'peak_hour', 'evening_hour',
    ],
)
all_profiles = data[data['atlas_table'].eq('f3_province_season_hourly_shift')].copy()
profiles = all_profiles[all_profiles['season'].eq('Autumn')].copy()
provinces = sorted(profiles['province_en'].dropna().unique())

ncols = 4
nrows = int(np.ceil((len(provinces) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12.2, 16.0), sharex=True)
axes = np.asarray(axes).ravel()

for panel_index, (ax, province) in enumerate(zip(axes, provinces)):
    frame = profiles[profiles['province_en'].eq(province)].sort_values('hour')
    hours = frame['hour'].to_numpy(dtype=int)
    absolute = frame[[
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h',
    ]].to_numpy(dtype=float)
    panel_scale = max(float(np.max(np.abs(absolute))), 1e-9)
    shift = frame['shifted_workload_gwh_it_h'].to_numpy(dtype=float) / panel_scale
    ax.bar(hours, shift, width=0.78, color=np.where(shift >= 0, GOLD, BLUE),
           alpha=0.76, edgecolor='none', zorder=2)
    ax.axhline(0, color=MUTED, lw=0.55)
    ax.set_xlim(-0.6, 23.6)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.grid(axis='both', color=GRID, lw=0.42, alpha=0.60)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=6.3, length=2.2, pad=1.4)
    ax.text(0.025, 0.955, province, transform=ax.transAxes, ha='left', va='top',
            fontsize=7.7, fontweight='bold', color=INK, zorder=8)
    ax.text(0.975, 0.955, f'peak={panel_scale:.2f} GWh h$^{-1}$',
            transform=ax.transAxes, ha='right', va='top', fontsize=5.8,
            color=MUTED, zorder=8)
    ax.plot(hours, frame['demand_static_gwh_h'] / panel_scale,
            color=MUTED, lw=1.05, ls='--')
    ax.plot(hours, frame['demand_shifted_gwh_h'] / panel_scale,
            color=INK, lw=1.30)
    ax.plot(hours, frame['clean_power_used_gwh_h'] / panel_scale,
            color=GREEN, lw=1.15)
    ax.plot(hours, frame['routed_compute_gwh_it_h'] / panel_scale,
            color=AMBER, lw=1.05, ls=':')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(SPINE)
        spine.set_linewidth(0.80)

legend_ax = axes[len(provinces)]
legend_ax.axis('off')
legend_ax.legend(handles=[
    Patch(facecolor=GOLD, alpha=0.76, label='Shifted into hour'),
    Patch(facecolor=BLUE, alpha=0.76, label='Shifted out of hour'),
    mpl.lines.Line2D([0], [0], color=MUTED, lw=1.2, ls='--', label='Static IT workload'),
    mpl.lines.Line2D([0], [0], color=INK, lw=1.4, label='After temporal shifting'),
    mpl.lines.Line2D([0], [0], color=GREEN, lw=1.3, label='Clean power used'),
    mpl.lines.Line2D([0], [0], color=AMBER, lw=1.2, ls=':', label='Routed compute'),
], loc='center', frameon=False, fontsize=8.1, ncol=1)
for ax in axes[len(provinces) + 1:]:
    ax.set_visible(False)

pass
fig.supxlabel('Clock hour', fontsize=9.0, y=0.006)
fig.supylabel('Normalized value (panel peak = 1)', fontsize=9.0, x=0.006)
fig.tight_layout(rect=(0.025, 0.022, 0.998, 0.982), h_pad=0.82, w_pad=0.65)
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=300, facecolor='white', bbox_inches='tight')
plt.close(fig)

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "province_hourly_dispatch_winter.png"

INK = '#27364A'
MUTED = '#8B95A1'
GOLD = '#D89C45'
BLUE = '#5B7DB8'
GREEN = '#357B43'
AMBER = '#C58900'
GRID = '#DCE4EA'
SPINE = '#5F6873'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 7.4,
    'axes.linewidth': 0.8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

data = pd.read_feather(
    DATA,
    columns=[
        'atlas_table', 'province_en', 'season', 'hour',
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h', 'peak_hour', 'evening_hour',
    ],
)
all_profiles = data[data['atlas_table'].eq('f3_province_season_hourly_shift')].copy()
profiles = all_profiles[all_profiles['season'].eq('Winter')].copy()
provinces = sorted(profiles['province_en'].dropna().unique())

ncols = 4
nrows = int(np.ceil((len(provinces) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12.2, 16.0), sharex=True)
axes = np.asarray(axes).ravel()

for panel_index, (ax, province) in enumerate(zip(axes, provinces)):
    frame = profiles[profiles['province_en'].eq(province)].sort_values('hour')
    hours = frame['hour'].to_numpy(dtype=int)
    absolute = frame[[
        'demand_static_gwh_h', 'demand_shifted_gwh_h',
        'clean_power_used_gwh_h', 'routed_compute_gwh_it_h',
        'shifted_workload_gwh_it_h',
    ]].to_numpy(dtype=float)
    panel_scale = max(float(np.max(np.abs(absolute))), 1e-9)
    shift = frame['shifted_workload_gwh_it_h'].to_numpy(dtype=float) / panel_scale
    ax.bar(hours, shift, width=0.78, color=np.where(shift >= 0, GOLD, BLUE),
           alpha=0.76, edgecolor='none', zorder=2)
    ax.axhline(0, color=MUTED, lw=0.55)
    ax.set_xlim(-0.6, 23.6)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.grid(axis='both', color=GRID, lw=0.42, alpha=0.60)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=6.3, length=2.2, pad=1.4)
    ax.text(0.025, 0.955, province, transform=ax.transAxes, ha='left', va='top',
            fontsize=7.7, fontweight='bold', color=INK, zorder=8)
    ax.text(0.975, 0.955, f'peak={panel_scale:.2f} GWh h$^{-1}$',
            transform=ax.transAxes, ha='right', va='top', fontsize=5.8,
            color=MUTED, zorder=8)
    ax.plot(hours, frame['demand_static_gwh_h'] / panel_scale,
            color=MUTED, lw=1.05, ls='--')
    ax.plot(hours, frame['demand_shifted_gwh_h'] / panel_scale,
            color=INK, lw=1.30)
    ax.plot(hours, frame['clean_power_used_gwh_h'] / panel_scale,
            color=GREEN, lw=1.15)
    ax.plot(hours, frame['routed_compute_gwh_it_h'] / panel_scale,
            color=AMBER, lw=1.05, ls=':')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(SPINE)
        spine.set_linewidth(0.80)

legend_ax = axes[len(provinces)]
legend_ax.axis('off')
legend_ax.legend(handles=[
    Patch(facecolor=GOLD, alpha=0.76, label='Shifted into hour'),
    Patch(facecolor=BLUE, alpha=0.76, label='Shifted out of hour'),
    mpl.lines.Line2D([0], [0], color=MUTED, lw=1.2, ls='--', label='Static IT workload'),
    mpl.lines.Line2D([0], [0], color=INK, lw=1.4, label='After temporal shifting'),
    mpl.lines.Line2D([0], [0], color=GREEN, lw=1.3, label='Clean power used'),
    mpl.lines.Line2D([0], [0], color=AMBER, lw=1.2, ls=':', label='Routed compute'),
], loc='center', frameon=False, fontsize=8.1, ncol=1)
for ax in axes[len(provinces) + 1:]:
    ax.set_visible(False)

pass
fig.supxlabel('Clock hour', fontsize=9.0, y=0.006)
fig.supylabel('Normalized value (panel peak = 1)', fontsize=9.0, x=0.006)
fig.tight_layout(rect=(0.025, 0.022, 0.998, 0.982), h_pad=0.82, w_pad=0.65)
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=300, facecolor='white', bbox_inches='tight')
plt.close(fig)

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "annual_hourly_route_atlas_24h.png"

PALETTE = {
    "flow": "#174C7C",
    "outer": "#467CB9",
    "inner": "#8FB9B2",
    "boundary": "#4B5560",
    "ink": "#1F2D3A",
    "muted": "#667786",
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 6.5,
    "axes.linewidth": 0.5,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

REQUIRED_COLUMNS = [
    "atlas_table",
    "record_type",
    "segment_id",
    "point_id",
    "map_x",
    "map_y",
    "xmin",
    "xmax",
    "ymin",
    "ymax",
    "province",
    "province_en",
    "source",
    "target",
    "hour",
    "flow_gwh_it_h",
    "executed_supply_twh_it",
    "local_processed_twh_it",
]
data = pd.read_feather(DATA, columns=REQUIRED_COLUMNS)
data = data[data["atlas_table"].eq("f2_annual_hourly_route_atlas_24")].copy()
boundary = data[data["record_type"].eq("boundary")].copy()
bounds = data[data["record_type"].eq("bounds")].iloc[0]
coords = data[data["record_type"].eq("coords")].copy()
service = data[data["record_type"].eq("service")].copy()
flows = data[data["record_type"].eq("flow")].copy()

known = set(coords["province"].dropna())
flows = flows[flows["source"].isin(known) & flows["target"].isin(known)].copy()
coord_lookup = coords.set_index("province")

GLOBAL_MAX_FLOW = max(float(flows["flow_gwh_it_h"].max()), 1e-12)
GLOBAL_MAX_NODE = max(float(service["executed_supply_twh_it"].max()), 1e-12)
FLOW_WIDTH_AT_GLOBAL_MAX = 5.6

service = service.drop(columns=["map_x", "map_y", "lon", "lat"], errors="ignore")
service = service.merge(
    coords[["province", "province_en", "map_x", "map_y"]],
    on=["province", "province_en"],
    how="left",
).dropna(subset=["map_x", "map_y"])
total = np.maximum(service["executed_supply_twh_it"].astype(float), 0.0)
local = np.maximum(service["local_processed_twh_it"].astype(float), 0.0)
outer_size = 4.0 + 78.0 * np.sqrt(total / GLOBAL_MAX_NODE)
inner_size = outer_size * np.where(total > 0, local / total, 0.0)

segments = []
for _, segment in boundary.groupby("segment_id"):
    segment = segment.sort_values("point_id")
    segments.append(segment[["map_x", "map_y"]].to_numpy(dtype=float))

pairs = sorted(set(zip(flows["source"], flows["target"])))
pair_curve = {
    pair: (0.065 if index % 2 == 0 else -0.065)
    for index, pair in enumerate(pairs)
}

fig, axes = plt.subplots(5, 5, figsize=(14.2, 15.5))
axes = np.asarray(axes).ravel()

for hour, ax in enumerate(axes):
    if hour >= 24:
        ax.axis("off")
        continue
    ax.set_xlim(float(bounds["xmin"]), float(bounds["xmax"]))
    ax.set_ylim(float(bounds["ymin"]), float(bounds["ymax"]))
    ax.set_aspect("equal", adjustable="box", anchor="C")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.add_collection(LineCollection(
        segments,
        colors=PALETTE["boundary"],
        linewidths=0.95,
        alpha=1.0,
        zorder=0,
    ))

    hourly = flows[flows["hour"].eq(hour)].sort_values("flow_gwh_it_h")
    for _, row in hourly.iterrows():
        source = coord_lookup.loc[row["source"]]
        target = coord_lookup.loc[row["target"]]
        strength = float(row["flow_gwh_it_h"]) / GLOBAL_MAX_FLOW
        ax.add_patch(FancyArrowPatch(
            (source["map_x"], source["map_y"]),
            (target["map_x"], target["map_y"]),
            arrowstyle="-|>",
            connectionstyle=f"arc3,rad={pair_curve[(row['source'], row['target'])]}",
            mutation_scale=2.8 + 5.5 * np.sqrt(strength),
            linewidth=FLOW_WIDTH_AT_GLOBAL_MAX * strength,
            color=PALETTE["flow"],
            alpha=0.13 + 0.67 * np.sqrt(strength),
            zorder=2,
        ))

    ax.scatter(
        service["map_x"], service["map_y"],
        s=outer_size,
        c=PALETTE["outer"],
        alpha=0.25,
        edgecolor="white",
        linewidth=0.42,
        zorder=3,
    )
    ax.scatter(
        service["map_x"], service["map_y"],
        s=inner_size,
        c=PALETTE["inner"],
        alpha=0.48,
        edgecolor="white",
        linewidth=0.42,
        zorder=4,
    )

    total_hour = float(hourly["flow_gwh_it_h"].sum())
    ax.set_title(
        f"{hour:02d}:00   {total_hour:.2f} GWh-IT h$^{{-1}}$",
        loc="left",
        fontsize=6.8,
        fontweight="bold",
        color=PALETTE["ink"],
        pad=1.5,
    )

reference_values = np.asarray([0.25, 0.50, 1.00]) * GLOBAL_MAX_FLOW
flow_handles = [
    Line2D(
        [0], [0],
        color=PALETTE["flow"],
        linewidth=FLOW_WIDTH_AT_GLOBAL_MAX * value / GLOBAL_MAX_FLOW,
        alpha=0.62,
        label=f"{value:.2f} GWh-IT h$^{{-1}}$",
    )
    for value in reference_values
]
node_handles = [
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=PALETTE["outer"],
           markeredgecolor="white", alpha=0.5, markersize=7.0,
           label="Total executed workload"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=PALETTE["inner"],
           markeredgecolor="white", alpha=0.7, markersize=5.0,
           label="Local retained workload"),
]
fig.legend(
    handles=flow_handles + node_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.008),
    ncol=5,
    frameon=False,
    fontsize=7.0,
    handlelength=2.8,
    columnspacing=1.5,
)
pass
pass
fig.subplots_adjust(
    left=0.008,
    right=0.996,
    bottom=0.040,
    top=0.966,
    wspace=-0.20,
    hspace=0.015,
)
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=600, facecolor="white", bbox_inches="tight", pad_inches=0.03)
plt.close(fig)

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary Figure 8

In [ ]:
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch

ROOT = Path.cwd()
DATA = ROOT / "data/rf_data.feather"
OUT = ROOT / "figures" / "panels" / "supp_route_formation_diagnostics.png"

FEATURES = [
    "distance_km",
    "same_region",
    "source_access_ratio",
    "source_access_clean_twh",
    "source_hydro_twh",
    "source_wind_solar_twh",
    "source_effective_pue",
    "source_carbon_intensity",
    "source_host_cap_mw",
    "target_demand_twh",
    "target_shortage_twh",
    "target_access_ratio",
]

LABELS = {
    "distance_km": "Road-network distance",
    "same_region": "Same-region link",
    "source_access_ratio": "Source clean access ratio",
    "source_access_clean_twh": "Source accessible clean power",
    "source_hydro_twh": "Source hydropower",
    "source_wind_solar_twh": "Source wind + solar",
    "source_effective_pue": "Source effective PUE",
    "source_carbon_intensity": "Source carbon intensity",
    "source_host_cap_mw": "Source hosting capacity",
    "target_demand_twh": "Target facility demand",
    "target_shortage_twh": "Target clean-power shortage",
    "target_access_ratio": "Target clean access ratio",
}

INK = "#243442"
MUTED = "#6C7C89"
GRID = "#DCE4E8"
BLUE = "#3F6F9F"
TEAL = "#278C7B"
GOLD = "#D39B35"
CORAL = "#C76556"
NETWORK = "#526D82"

def panel_label(ax, label: str, y_offset: float = 1.08):
    ax.text(
        -0.08,
        y_offset,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.3,
        fontweight="bold",
        color=INK,
    )

def main():
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
            "font.size": 7.4,
            "axes.labelsize": 7.5,
            "xtick.labelsize": 6.7,
            "ytick.labelsize": 6.7,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.65,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
        }
    )
    required = ["atlas_table", "active", "source", "target", *FEATURES]
    data = table("rf_road_pairs").assign(atlas_table="deep_route_pair_features_all")[required]
    frame = (
        data[data["atlas_table"].eq("deep_route_pair_features_all")]
        .dropna(subset=["active", "source", "target", *FEATURES])
        .reset_index(drop=True)
    )

    metrics = table('rf_metrics')
    evaluations = {row['validation']: {'auc': row['roc_auc'],
                   'ap': row['average_precision'], 'brier': row['brier_score']}
                   for row in metrics.to_dict('records')}
    importance = table('rf_importance')

    fig = plt.figure(figsize=(11.8, 6.6))
    gs = fig.add_gridspec(2, 6, hspace=0.38, wspace=0.55)
    ax_roc = fig.add_subplot(gs[0, 0:2])
    ax_pr = fig.add_subplot(gs[0, 2:4])
    ax_cal = fig.add_subplot(gs[0, 4:6])
    ax_imp = fig.add_subplot(gs[1, 0:2])
    response_gs = gs[1, 2:4].subgridspec(2, 2, hspace=0.55, wspace=0.48)
    response_axes = [fig.add_subplot(response_gs[i, j]) for i in range(2) for j in range(2)]
    ax_gate = fig.add_subplot(gs[1, 4:6])

    styles = {
        "pair": (BLUE, "Random candidate-pair CV"),
        "source": (TEAL, "Held-out source provinces"),
        "target": (CORAL, "Held-out target provinces"),
    }
    for mode, result in evaluations.items():
        colour, label = styles[mode]
        curve = table("rf_roc").query("validation == @mode")
        fpr, tpr = curve["fpr"], curve["tpr"]
        ax_roc.plot(fpr, tpr, color=colour, lw=1.7, label=f"{label} ({result['auc']:.2f})")
        curve = table("rf_pr").query("validation == @mode")
        precision, recall = curve["precision"], curve["recall"]
        ax_pr.plot(recall, precision, color=colour, lw=1.7, label=f"{label} ({result['ap']:.2f})")
    ax_roc.plot([0, 1], [0, 1], color="#AEB8BF", lw=0.8, ls="--")
    ax_roc.set(xlabel="False-positive rate", ylabel="True-positive rate", xlim=(0, 1), ylim=(0, 1))
    ax_pr.axhline(frame["active"].mean(), color="#AEB8BF", lw=0.8, ls="--")
    ax_pr.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
    ax_roc.legend(frameon=False, fontsize=6.1, loc="lower right")
    ax_pr.legend(frameon=False, fontsize=6.1, loc="upper right")
    panel_label(ax_roc, "a")
    panel_label(ax_pr, "b")

    means = importance.mean().sort_values(ascending=False)
    selected = means.head(9).index[::-1]
    yy = np.arange(len(selected))
    colours = [NETWORK if x in {"distance_km", "same_region"} else TEAL if x.startswith("source_") else CORAL for x in selected]
    for y_pos, feature, colour in zip(yy, selected, colours):
        vals = importance[feature].to_numpy()
        ax_imp.hlines(y_pos, vals.min(), vals.max(), color=colour, lw=1.1, alpha=0.55)
        ax_imp.scatter(vals, np.full_like(vals, y_pos), s=13, color=colour, alpha=0.5, zorder=3)
        ax_imp.scatter(vals.mean(), y_pos, s=38, color=colour, edgecolor="white", lw=0.6, zorder=4)
    ax_imp.axvline(0, color=INK, lw=0.65)
    ax_imp.set_yticks(yy, [LABELS[x] for x in selected])
    ax_imp.set_xlabel("Held-out permutation importance (ROC-AUC loss)")
    ax_imp.grid(axis="x", color=GRID, lw=0.5)
    panel_label(ax_imp, "d")

    cal = table('rf_calibration')
    ax_cal.plot([0, 1], [0, 1], color="#AEB8BF", lw=0.8, ls="--")
    ax_cal.plot(cal["pred"], cal["obs"], color=BLUE, marker="o", ms=3.6, lw=1.5)
    ax_cal.set(xlabel="Predicted activation probability", ylabel="Observed activation rate", xlim=(0, 1), ylim=(0, 1))
    ax_cal.text(0.03, 0.94, f"Brier score {evaluations['pair']['brier']:.3f}", transform=ax_cal.transAxes, va="top", color=MUTED)
    panel_label(ax_cal, "c")

    response_features = ["distance_km", "source_access_ratio", "target_shortage_twh", "source_carbon_intensity"]
    response_titles = [
        "Road-network distance (km)",
        "Source clean access ratio",
        "Target shortage (TWh)",
        "Source carbon intensity\n(tCO$_2$ MWh-IT$^{-1}$)",
    ]
    for ax, feature, title in zip(response_axes, response_features, response_titles):
        tab = table("rf_response").query("feature == @feature")
        ax.fill_between(tab["x"], tab["lo"], tab["hi"], color=GRID, alpha=0.9, lw=0)
        ax.plot(tab["x"], tab["observed"], color=INK, marker="o", ms=2.6, lw=1.1, label="Observed")
        ax.plot(tab["x"], tab["predicted"], color=GOLD, lw=1.3, label="CV prediction")
        ax.set_xlabel(title, fontsize=6.9, labelpad=2)
        ax.set_ylim(0, max(0.55, float(tab["hi"].max()) * 1.08))
        ax.grid(axis="y", color=GRID, lw=0.45)
    response_axes[0].set_ylabel("Activation rate")
    response_axes[2].set_ylabel("Activation rate")
    response_axes[0].legend(frameon=False, fontsize=5.7, loc="upper right")
    panel_label(response_axes[0], "e", y_offset=1.16)

    gate = table("rf_gate").pivot(index="shortage_bin", columns="source_bin", values="rate")
    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "route_gate", ["#F4F7F8", "#BFDCD4", "#4FA995", "#176B64"]
    )
    row_order = ["High", "Medium", "Low", "None"]
    col_order = ["Q1", "Q2", "Q3", "Q4", "Q5"]
    for row, shortage in enumerate(row_order):
        for col, source_bin in enumerate(col_order):
            value = float(gate.loc[shortage, source_bin])
            patch = FancyBboxPatch(
                (col - 0.44, row - 0.39),
                0.88,
                0.78,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                facecolor=cmap(value / 0.65),
                edgecolor="white",
                linewidth=1.0,
            )
            ax_gate.add_patch(patch)
            text_colour = "white" if value > 0.38 else INK
            ax_gate.text(
                col,
                row,
                f"{value * 100:.0f}%",
                ha="center",
                va="center",
                fontsize=7.0,
                color=text_colour,
                fontweight="bold" if value > 0.35 else "normal",
            )
    ax_gate.set_xlim(-0.55, 4.55)
    ax_gate.set_ylim(3.55, -0.55)
    ax_gate.set_xticks(range(5))
    ax_gate.set_xticklabels(["lowest", "Q2", "Q3", "Q4", "highest"])
    ax_gate.set_yticks(range(4))
    ax_gate.set_yticklabels(row_order)
    ax_gate.set_xlabel("Source clean-power access ratio")
    ax_gate.set_ylabel("Target clean-power shortage")
    ax_gate.tick_params(length=0)
    for spine in ax_gate.spines.values():
        spine.set_visible(False)
    panel_label(ax_gate, "f")

    OUT.parent.mkdir(parents=True, exist_ok=True)
    metrics = pd.DataFrame([dict(validation=mode, roc_auc=v['auc'], average_precision=v['ap'], brier_score=v['brier'])
                            for mode, v in evaluations.items()])
    metrics.to_csv(OUT.parent / 'rf_road_metrics.csv', index=False)
    importance.to_csv(OUT.parent / 'rf_road_importance_by_fold.csv', index=False)
    (OUT.parent / 'figure_recipe_QA.json').write_text(json.dumps({
        'approval_status': 'RF_APPROVED_KAPPA_PENDING', 'frozen_RF': True, 'retrained': False, 'figure_size_inches': list(fig.get_size_inches()),
        'feature_count': len(FEATURES), 'features': FEATURES, 'only_input_change': 'distance_km',
        'n_estimators': 300, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'class_weight': 'balanced',
        'cv_folds': 5, 'permutation_repeats': 12, 'original_template_style_retained': True,
        'no_LP_called': True, 'no_feature_ablation': True,
        'carbon_intensity_unit': 'tCO2 per MWh-IT',
        'figure_title_moved_to_caption': True,
        'panel_titles': 'letters only'}, indent=2), encoding='utf-8')
    fig.savefig(OUT, dpi=350, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    pass
    pass
    for mode, result in evaluations.items():
        pass

if __name__ == "__main__":
    with mpl.rc_context(mpl.rcParamsDefault):
        mpl.rcParams.update({'axes.titlesize': 9, 'legend.fontsize': 7,
                             'svg.fonttype': 'none', 'pdf.fonttype': 42})
        main()

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "supp_distance_routing_friction.png"

INK = "#263746"
MUTED = "#6D7D89"
GRID = "#DCE4E8"
BLUE = "#4C79A8"
TEAL = "#2A8E7E"
GOLD = "#D59B32"
LIGHT = "#CAD7DF"

def wilson(successes, totals):
    z = 1.96
    p = successes / totals
    denominator = 1 + z**2 / totals
    centre = (p + z**2 / (2 * totals)) / denominator
    half = z * np.sqrt((p * (1 - p) + z**2 / (4 * totals)) / totals) / denominator
    return centre - half, centre + half

def panel_label(ax, label):
    ax.text(-0.12, 1.06, label, transform=ax.transAxes, ha="left", va="bottom",
            fontweight="bold", fontsize=12, color=INK)

def main():
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
            "font.size": 7.0,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.linewidth": 0.7,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
        }
    )
    columns = ["atlas_table", "active", "distance_km", "flow_twh_it"]
    data = pd.read_feather(DATA, columns=columns)
    frame = data[data["atlas_table"].eq("deep_route_pair_features_all")].copy()
    edges = [0, 500, 1000, 1500, 2000, 2500, 3000, 4000]
    labels = ["<500", "500–1,000", "1,000–1,500", "1,500–2,000", "2,000–2,500", "2,500–3,000", ">3,000"]
    frame["distance_band"] = pd.cut(frame["distance_km"], edges, labels=labels, right=False)
    summary = (
        frame.groupby("distance_band", observed=True)
        .agg(
            candidates=("active", "size"),
            active_routes=("active", "sum"),
            activation_rate=("active", "mean"),
            routed_flow_twh_it=("flow_twh_it", "sum"),
        )
        .reindex(labels)
        .reset_index()
    )
    summary["inactive_routes"] = summary["candidates"] - summary["active_routes"]
    summary["flow_share"] = summary["routed_flow_twh_it"] / summary["routed_flow_twh_it"].sum()
    summary["cumulative_flow_share"] = summary["flow_share"].cumsum()
    summary["lo"], summary["hi"] = wilson(
        summary["active_routes"].to_numpy(), summary["candidates"].to_numpy()
    )

    fig = plt.figure(figsize=(7, 6.3))
    gs = fig.add_gridspec(2, 4, wspace=0.5, hspace=0.55)
    axes = [
        fig.add_subplot(gs[0, 0:2]),
        fig.add_subplot(gs[0, 2:4]),
        fig.add_subplot(gs[1, 1:3]),
    ]

    x = np.arange(len(summary))

    ax = axes[0]
    ax.bar(x, summary["candidates"], color=LIGHT, width=0.72, label="Candidate pairs")
    ax.bar(x, summary["active_routes"], color=BLUE, width=0.72, label="Active routes")
    ax.set_ylabel("Number of ordered province pairs")
    ax.legend(frameon=False, fontsize=6.0, loc="upper right")
    panel_label(ax, "a")

    ax = axes[1]
    rate = summary["activation_rate"].to_numpy()
    ax.fill_between(x, summary["lo"], summary["hi"], color="#C8DDD8", alpha=0.85)
    ax.plot(x, rate, color=TEAL, marker="o", ms=4.3, lw=1.7)
    ax.axhline(frame["active"].mean(), color=MUTED, ls="--", lw=0.8, label="Network-wide prevalence")
    ax.set_ylim(0, max(0.32, float(summary["hi"].max()) * 1.08))
    ax.set_ylabel("Route activation rate")
    ax.legend(frameon=False, fontsize=5.8, loc="upper right")
    panel_label(ax, "b")

    ax = axes[2]
    bars = ax.bar(x, summary["flow_share"] * 100, color=GOLD, width=0.68)
    ax.set_ylabel("Share of routed compute (%)")
    ax2 = ax.twinx()
    ax2.plot(x, summary["cumulative_flow_share"] * 100, color=INK, marker="o", ms=3.5, lw=1.4)
    ax2.set_ylim(0, 105)
    ax2.set_ylabel("Cumulative share (%)", color=INK)
    ax2.spines["right"].set_visible(True)
    for bar, value in zip(bars, summary["flow_share"] * 100):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.7, f"{value:.0f}%",
                ha="center", va="bottom", fontsize=5.6, color=INK)
    panel_label(ax, "c")

    for ax in axes:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_xlabel("Interprovincial distance (km)")
        ax.grid(axis="y", color=GRID, lw=0.5)
        ax.set_axisbelow(True)

    fig.subplots_adjust(top=0.95, bottom=0.12)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT, dpi=350, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    pass

    pass
    pass

if __name__ == "__main__":
    main()

from IPython.display import Image, display
display(Image(filename=str(OUT)))


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "main_figure5_e.png"
OUT_SVG = ROOT / "figures" / "panels" / "main_figure5_e.svg"

SOLAR = "#E6AA36"
WIND = "#477DB1"
INK = "#273444"
GRID = "#DCE4EA"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 7.2,
    "axes.titlesize": 8.0,
    "axes.labelsize": 7.1,
    "xtick.labelsize": 6.3,
    "ytick.labelsize": 6.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.7,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "savefig.facecolor": "white",
})

cols = [
    "atlas_table", "month",
    "absorbed_wind_potential_twh", "absorbed_solar_potential_twh",
    "curtailable_wind_potential_twh", "curtailable_solar_potential_twh",
]
data = pd.read_feather(DATA, columns=cols)
overlap = data[data["atlas_table"].eq("f5_temporal_curtailment_overlap")].copy()
if overlap.empty:
    raise ValueError("Missing atlas_table: f5_temporal_curtailment_overlap")
monthly = overlap.groupby("month", as_index=False)[[
    "absorbed_wind_potential_twh", "absorbed_solar_potential_twh",
    "curtailable_wind_potential_twh", "curtailable_solar_potential_twh",
]].sum()
monthly["month"] = monthly["month"].astype(int)

fig, ax = plt.subplots(figsize=(4.15, 3.67), dpi=300)
ax.bar(monthly["month"], monthly["absorbed_wind_potential_twh"], color=WIND, width=0.72, label="Wind potential use")
ax.bar(monthly["month"], monthly["absorbed_solar_potential_twh"], bottom=monthly["absorbed_wind_potential_twh"], color=SOLAR, width=0.72, label="Solar potential use")
ax.set(xlabel="Month", ylabel="Monthly potential use (TWh)", xticks=[1, 4, 7, 10])
ax.grid(axis="y", color=GRID, lw=0.55, zorder=0)
ax.set_axisbelow(True)
ax.set_title('a', loc="left", fontweight="bold", color=INK, pad=5)

ax2 = ax.twinx()
ax2.plot(monthly["month"], monthly["curtailable_wind_potential_twh"], color=WIND, marker="o", ms=2.5, lw=1.15, ls="--", alpha=0.72, label="Wind underused supply")
ax2.plot(monthly["month"], monthly["curtailable_solar_potential_twh"], color=SOLAR, marker="s", ms=2.4, lw=1.15, ls="--", alpha=0.78, label="Solar underused supply")
ax2.set_ylabel("Potentially underused supply (TWh)")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(True)

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles + handles2, labels + labels2, loc="upper left", frameon=False, fontsize=5.7, ncol=2, handlelength=1.5, columnspacing=0.75)
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(pad=0.55)
fig.savefig(OUT, dpi=300, facecolor="white")
fig.savefig(OUT_SVG, facecolor="white")
plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "supplement_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "supp_curtailment_province_month_heatmap.png"
OUT_SVG = OUT_DIR / "supp_curtailment_province_month_heatmap.svg"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 7.2,
    "axes.titlesize": 8.0,
    "axes.labelsize": 7.1,
    "xtick.labelsize": 6.3,
    "ytick.labelsize": 6.3,
    "axes.linewidth": 0.75,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

cols = ["atlas_table", "province_en", "month", "total_absorbed_potential_twh"]
data = pd.read_feather(DATA, columns=cols)
overlap = data[data["atlas_table"].eq("f5_temporal_curtailment_overlap")].copy()
if overlap.empty:
    raise ValueError("Missing atlas_table: f5_temporal_curtailment_overlap")
overlap["month"] = overlap["month"].astype(int)
heat = overlap.pivot_table(index="province_en", columns="month", values="total_absorbed_potential_twh", aggfunc="sum", fill_value=0.0)
heat = heat.reindex(columns=list(range(1, 13)), fill_value=0.0)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]

fig_h = 3.67
fig, ax = plt.subplots(figsize=(3, fig_h), dpi=300)
vals = heat.to_numpy(dtype=float)
vmax = max(float(np.nanpercentile(vals, 98)), float(vals.max()), 1e-6)
im = ax.imshow(vals, aspect="auto", cmap="YlOrRd", vmin=0, vmax=vmax)
ax.set_xticks(np.arange(12))
ax.set_xticklabels([str(i) for i in range(1, 13)])
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index.tolist())
ax.set_xlabel("Month")
ax.set_ylabel("Province")
ax.set_title('b', loc="left", fontweight="bold", pad=6)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.set_xticks(np.arange(-0.5, 12, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(heat.index), 1), minor=True)
ax.grid(which="minor", color="white", lw=0.35)
ax.tick_params(which="minor", bottom=False, left=False)
cbar = fig.colorbar(im, ax=ax, fraction=0.034, pad=0.018)
cbar.set_label("Potential use (TWh)", fontsize=7.1)
cbar.ax.tick_params(labelsize=6.3, length=2)
fig.tight_layout(pad=0.65)
fig.savefig(OUT, dpi=300, facecolor="white")
fig.savefig(OUT_SVG, facecolor="white")
plt.show()

pass
